# El Espectro Modular de $\pi$: Validación y Dualidad Computacional
**Autor:** José Ignacio Peinador Sala

Este cuaderno complementa el artículo *"El Espectro Modular de $\pi$: De la Estructura de Canales Primos a las Supercongruencias Elípticas"*. Se divide en dos arquitecturas complementarias que demuestran la tesis de la dualidad computacional:

1. **Validación Formal (Régimen de Baja Energía - Lean 4):** Demostración rigurosa del Lema del Filtro de Ruido. Se prueba deductivamente que las clases de residuo $\mathcal{C}_0, \mathcal{C}_2, \mathcal{C}_3, \mathcal{C}_4$ módulo 6 son divisibles por 2 o 3, aislando la estructura de los primos en los canales $6k \pm 1$.
2. **Matemática Experimental (Régimen Local - Python):** Implementación del algoritmo BBP Spigot para extraer dígitos hexadecimales aislados explotando la estructura modular holográfica.

## Parte I: Formalización del Sustrato Aritmético
Configuramos la infraestructura de Lean 4 e inyectamos el compilador en el entorno de Python.

In [1]:
%%bash
# Instalación del gestor de Lean 4
curl https://raw.githubusercontent.com/leanprover/elan/master/elan-init.sh -sSf | sh -s -- -y

info: downloading installer
info: default toolchain set to 'stable'


In [2]:
import os
# Añadir los binarios de Lean al PATH global de Colab
os.environ['PATH'] = "/root/.elan/bin:" + os.environ['PATH']
os.chdir('/content')
print("Entorno Lean 4 configurado.")

Entorno Lean 4 configurado.


### Demostración: El Filtro de Ruido de los Canales Primos
Generamos el proyecto matemático y demostramos el lema fundamental de la Sección 2.1 del artículo. Lean 4 validará que cualquier índice fuera de los canales $\mathcal{C}_1$ y $\mathcal{C}_5$ colapsa invariablemente en divisibilidad por 2 o por 3.

In [3]:
%%bash
# 1. Limpieza y creación del proyecto
rm -rf espectro_pi
lake new espectro_pi math
cd espectro_pi

# 2. Creación del archivo de demostración
cat << 'EOF' > EspectroPi.lean
import Mathlib

/--
  Lema del Filtro de Ruido (Sección 2.1):
  Cualquier número natural n cuyas clases de congruencia módulo 6
  sean 0, 2, 3, o 4, es necesariamente divisible por 2 o por 3.
  Esto demuestra que la información no trivial reside únicamente en 6k+1 y 6k+5.
-/
theorem noise_filter (n : ℕ) (h : n % 6 = 0 ∨ n % 6 = 2 ∨ n % 6 = 3 ∨ n % 6 = 4) :
  2 ∣ n ∨ 3 ∣ n := by
  rcases h with h0 | h2 | h3 | h4
  · left
    exact ⟨3 * (n / 6), by omega⟩
  · left
    exact ⟨3 * (n / 6) + 1, by omega⟩
  · right
    exact ⟨2 * (n / 6) + 1, by omega⟩
  · left
    exact ⟨3 * (n / 6) + 2, by omega⟩

EOF

# 3. Hidratación y validación formal
echo "Descargando la caché de Mathlib (puede tardar un minuto)..."
lake update > /dev/null 2>&1
lake exe cache get! > /dev/null 2>&1

echo "Validando el lema geométrico con Lean 4..."
lake build

installing leantar 0.1.17
Fetching ProofWidgets cloud release... done!
Current branch: HEAD
Using cache (Azure) from origin: leanprover-community/mathlib4
Attempting to download 8232 file(s) from leanprover-community/mathlib4 cache
Decompressed 8232 file(s)
Already decompressed 8232 file(s)
Descargando la caché de Mathlib (puede tardar un minuto)...
Validando el lema geométrico con Lean 4...
⚠ [8248/8249] Built EspectroPi (197s)

Note: This linter can be disabled with `set_option linter.style.docString false`
Build completed successfully (8249 jobs).


info: downloading https://releases.lean-lang.org/lean4/v4.29.1/lean-4.29.1-linux.tar.zst
info: installing /root/.elan/toolchains/leanprover--lean4---v4.29.1
info: espectro_pi: no previous manifest, creating one from scratch
info: leanprover-community/mathlib: cloning https://github.com/leanprover-community/mathlib4
info: leanprover-community/mathlib: checking out revision '5e932f97dd25535344f80f9dd8da3aab83df0fe6'
info: plausible: cloning https://github.com/leanprover-community/plausible
info: plausible: checking out revision '83e90935a17ca19ebe4b7893c7f7066e266f50d3'
info: LeanSearchClient: cloning https://github.com/leanprover-community/LeanSearchClient
info: LeanSearchClient: checking out revision 'c5d5b8fe6e5158def25cd28eb94e4141ad97c843'
info: importGraph: cloning https://github.com/leanprover-community/import-graph
info: importGraph: checking out revision '48d5698bc464786347c1b0d859b18f938420f060'
info: proofwidgets: cloning https://github.com/leanprover-community/ProofWidgets4
i

## Parte II: Dualidad Computacional y Algoritmos Spigot (Python)

Como se expone en la Sección 5 del artículo, la estructura modular de $\pi$ permite una extracción holográfica y local de información. A continuación implementamos el algoritmo de Bailey-Borwein-Plouffe (BBP) sugerido en el manuscrito.

*Nota técnica:* Para evitar errores de exponente negativo en la función modular `pow()` de Python, la serie se divide canónicamente en dos sumatorios: uno computado modularmente (para iteraciones $k \le n$) y uno analítico de punto flotante (para iteraciones $k > n$).

In [4]:
def bbp_spigot_pi(n):
    """
    Calcula el n-ésimo dígito hexadecimal de Pi explotando la estructura
    modular holográfica (basado en la fórmula BBP).
    """
    def term(j, n):
        # Parte 1: k <= n (Cálculo modular entero para retener precisión)
        s_left = 0.0
        for k in range(n + 1):
            denominador = 8 * k + j
            # pow(base, exp, mod) es la clave de la localidad
            numerador = pow(16, n - k, denominador)
            s_left += numerador / denominador

        # Parte 2: k > n (Suma infinita convergente en punto flotante)
        s_right = 0.0
        k = n + 1
        while True:
            denominador = 8 * k + j
            termino_infinito = (16.0 ** (n - k)) / denominador
            if termino_infinito < 1e-15:
                break
            s_right += termino_infinito
            k += 1

        return s_left + s_right

    # Suma sobre los canales de la base 16 dictada por la identidad BBP
    pi_hex_frac = 4 * term(1, n) - 2 * term(4, n) - 1 * term(5, n) - 1 * term(6, n)

    # Extraemos solo la parte fraccionaria escalar
    pi_hex_frac = pi_hex_frac - int(pi_hex_frac)
    if pi_hex_frac < 0:
        pi_hex_frac += 1

    # Convertimos la porción decimal resultante a su símbolo hexadecimal
    return hex(int(pi_hex_frac * 16))[2:].upper()

# --- Experimento de Validación ---
print("Validación de Extracción Local (Spigot BBP)")
print("-" * 45)
# Los primeros dígitos de pi en hexadecimal son: 3.243F6A8885A308D3...
# n=0 -> '2', n=1 -> '4', n=2 -> '3', n=3 -> 'F', etc.
indices_prueba = [0, 1, 2, 3, 4, 10]

for n in indices_prueba:
    digito = bbp_spigot_pi(n)
    print(f"Dígito hexadecimal en la posición {n:2d} : {digito}")

Validación de Extracción Local (Spigot BBP)
---------------------------------------------
Dígito hexadecimal en la posición  0 : 2
Dígito hexadecimal en la posición  1 : 4
Dígito hexadecimal en la posición  2 : 3
Dígito hexadecimal en la posición  3 : F
Dígito hexadecimal en la posición  4 : 6
Dígito hexadecimal en la posición 10 : A
